# 🚗 SIH — ADAS Vision: Drivable Area Segmentation (IDD)
**Model:** LRASPP MobileNetV3-Large | **Dataset:** IDD Segmentation (Indian Driving Dataset)  
**Team:** Krishna2077-s | **GPU:** T4 x2 (Kaggle free tier)

> ⚠️ Advisory / research only — never wire this to a vehicle's controls.

### Dataset structure confirmed
```
/kaggle/input/datasets/krishna22b/idd-segmentation/IDD_Segmentation/
  gtFine/         ← JSON polygon annotations
    train/  val/
  leftImg8bit/    ← images (.png)
    train/  val/  test/
```

In [ ]:
# ── Cell 1 : verify GPU ──────────────────────────────────────────────────────
import torch, os, sys

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {DEVICE}')

In [ ]:
# ── Cell 2 : install packages ────────────────────────────────────────────────
!pip install -q albumentations==1.4.3 onnx onnxruntime opencv-python-headless

In [ ]:
# ── Cell 3 : hardcoded paths (confirmed from diagnostic) ─────────────────────
import pathlib

IDD_ROOT  = pathlib.Path('/kaggle/input/datasets/krishna22b/idd-segmentation/IDD_Segmentation')
IMG_DIR   = IDD_ROOT / 'leftImg8bit'   # train/ val/ test/
ANN_DIR   = IDD_ROOT / 'gtFine'        # train/ val/

ADAS_DS   = pathlib.Path('/kaggle/input/datasets/krishna22b/adas-weights')
WEIGHTS   = ADAS_DS / 'drivable_idd_full_best.pth'

WORK_DIR  = pathlib.Path('/kaggle/working')
DATA_DIR  = WORK_DIR / 'data'
MODEL_DIR = WORK_DIR / 'models'
IMG_OUT   = DATA_DIR / 'images'
MSK_OUT   = DATA_DIR / 'masks'
for d in [IMG_OUT, MSK_OUT, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── verify ───────────────────────────────────────────────────────────────────
print(f'IDD root  exists : {IDD_ROOT.exists()}')
print(f'Images    exists : {IMG_DIR.exists()}')
print(f'Annots    exists : {ANN_DIR.exists()}')
print(f'Weights   exists : {WEIGHTS.exists()}')

FINE_TUNE = WEIGHTS.exists()

# ── hyper-parameters ─────────────────────────────────────────────────────────
IMG_W, IMG_H = 768, 432
BATCH_SIZE   = 8
EPOCHS       = 20
LR           = 3e-4
SUBSET       = None    # set e.g. 500 for a quick smoke-test
NUM_CLASSES  = 3       # 0=background  1=drivable  2=alt-drivable

print('\nConfig OK ✅')

In [ ]:
# ── Cell 4 : copy .py source files ───────────────────────────────────────────
import shutil

py_files = list(ADAS_DS.rglob('*.py'))
for f in py_files:
    shutil.copy2(f, WORK_DIR / f.name)
print(f'✅ Copied {len(py_files)} source files')

sys.path.insert(0, str(WORK_DIR))

In [ ]:
# ── Cell 5 : explore IDD structure ───────────────────────────────────────────
all_imgs = sorted(IMG_DIR.rglob('*_leftImg8bit.png'))
all_anns = sorted(ANN_DIR.rglob('*_gtFine_polygons.json'))

print(f'Total images      : {len(all_imgs)}')
print(f'Total annotations : {len(all_anns)}')
print(f'\nSample image : {all_imgs[0]}')
print(f'Sample annot : {all_anns[0]}')

# peek at one annotation to see label names
import json
with open(all_anns[0]) as f:
    sample = json.load(f)
labels_found = {obj['label'] for obj in sample.get('objects', [])}
print(f'\nLabels in sample: {sorted(labels_found)}')

In [ ]:
# ── Cell 6 : convert IDD JSON → single-channel mask PNGs ─────────────────────
# 0 = background   1 = drivable (road)   2 = alt-drivable (parking / sidewalk)
import json, cv2
import numpy as np
from PIL import Image

# IDD label names that map to our 3 classes
DRIVABLE     = {'road', 'drivable'}
ALT_DRIVABLE = {'parking', 'railtrack', 'drivable fallback'}

def convert_sample(img_path, ann_path):
    img = cv2.imread(str(img_path))
    if img is None: return False
    img = cv2.resize(img, (IMG_W, IMG_H))

    with open(ann_path) as f:
        ann = json.load(f)

    oh = ann.get('imgHeight', img.shape[0])
    ow = ann.get('imgWidth',  img.shape[1])
    sx, sy = IMG_W / ow, IMG_H / oh
    mask = np.zeros((IMG_H, IMG_W), dtype=np.uint8)

    for obj in ann.get('objects', []):
        lbl = obj.get('label', '').lower()
        if lbl in DRIVABLE:          cls = 1
        elif lbl in ALT_DRIVABLE:    cls = 2
        else:                        continue
        pts = np.array(
            [[int(x * sx), int(y * sy)] for x, y in obj['polygon']],
            dtype=np.int32
        )
        cv2.fillPoly(mask, [pts], cls)

    stem = img_path.stem.replace('_leftImg8bit', '')
    cv2.imwrite(str(IMG_OUT / f'{stem}.jpg'), img)
    cv2.imwrite(str(MSK_OUT / f'{stem}.png'), mask)
    return True

# match images ↔ annotations by stem
ann_map = {p.stem.replace('_gtFine_polygons', ''): p for p in all_anns}
pairs   = []
for ip in all_imgs[:SUBSET]:
    stem = ip.stem.replace('_leftImg8bit', '')
    if stem in ann_map:
        pairs.append((ip, ann_map[stem]))

print(f'Matched {len(pairs)} pairs — converting…')
ok = sum(convert_sample(i, j) for i, j in pairs)
print(f'✅ Done — {ok} pairs written')

# sanity: check mask has drivable pixels
sample_mask = np.array(Image.open(sorted(MSK_OUT.glob('*.png'))[0]))
print(f'Sample mask unique values : {np.unique(sample_mask)}  (should include 1)')

In [ ]:
# ── Cell 7 : augmentation ─────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.RandomResizedCrop(IMG_H, IMG_W, scale=(0.6, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.1), contrast_limit=0.2, p=1),
        A.RandomFog(fog_coef_lower=0.2, fog_coef_upper=0.6, p=1),
        A.MotionBlur(blur_limit=(7, 21), p=1),
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1, num_shadows_upper=3, p=1),
        A.GaussNoise(var_limit=(20, 80), p=1),
    ], p=0.7),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('Augmentation ready ✅')

In [ ]:
# ── Cell 8 : dataset + dataloaders ───────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader, random_split

class IDDDataset(Dataset):
    def __init__(self, img_dir, msk_dir, transform=None):
        self.imgs      = sorted(img_dir.glob('*.jpg'))
        self.masks     = [msk_dir / (p.stem + '.png') for p in self.imgs]
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img  = np.array(Image.open(self.imgs[idx]).convert('RGB'))
        mask = np.array(Image.open(self.masks[idx]))
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img, mask = out['image'], out['mask'].long()
        return img, mask

full_ds = IDDDataset(IMG_OUT, MSK_OUT)
n_val   = max(1, int(0.1 * len(full_ds)))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val])

train_ds.dataset.transform = train_transform
val_ds.dataset.transform   = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train}  |  Val: {n_val} ✅')

In [ ]:
# ── Cell 9 : model ────────────────────────────────────────────────────────────
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

model = lraspp_mobilenet_v3_large(num_classes=NUM_CLASSES)

if FINE_TUNE:
    print(f'Loading pretrained weights…')
    ckpt  = torch.load(WEIGHTS, map_location='cpu')
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state, strict=False)
    print('✅ Weights loaded (fine-tuning)')
else:
    print('Training from scratch')

model = model.to(DEVICE)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f'Using {torch.cuda.device_count()} GPUs')

print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f} M')

In [ ]:
# ── Cell 10 : loss + optimiser ────────────────────────────────────────────────
from torch import nn, optim

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def iou_score(pred, target):
    pred   = pred.argmax(1).cpu().numpy().ravel()
    target = target.cpu().numpy().ravel()
    ious   = []
    for c in range(NUM_CLASSES):
        tp = ((pred == c) & (target == c)).sum()
        fp = ((pred == c) & (target != c)).sum()
        fn = ((pred != c) & (target == c)).sum()
        if tp + fp + fn == 0: continue
        ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0

print('Loss & optimiser ready ✅')

In [ ]:
# ── Cell 11 : training loop ───────────────────────────────────────────────────
import time

best_iou  = 0.0
BEST_CKPT = MODEL_DIR / 'best.pth'
history   = {'train_loss': [], 'val_iou': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, tloss = time.time(), 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs)['out'], masks)
        loss.backward(); optimizer.step()
        tloss += loss.item()
    tloss /= len(train_loader)

    model.eval()
    viou = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            viou += iou_score(model(imgs)['out'], masks)
    viou /= len(val_loader)

    scheduler.step()
    history['train_loss'].append(tloss)
    history['val_iou'].append(viou)
    print(f'Epoch {epoch:03d}/{EPOCHS}  loss={tloss:.4f}  val_IoU={viou:.4f}  {time.time()-t0:.0f}s')

    if viou > best_iou:
        best_iou = viou
        m = model.module if hasattr(model, 'module') else model
        torch.save({'epoch': epoch, 'model_state_dict': m.state_dict(),
                    'val_iou': viou, 'optimizer_state_dict': optimizer.state_dict()}, BEST_CKPT)
        print(f'  ✅ Best saved  IoU={viou:.4f}')

print(f'\n🏁 Done!  Best val IoU: {best_iou:.4f}')

In [ ]:
# ── Cell 12 : training curves ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_loss'], 'r'); ax[0].set_title('Train Loss'); ax[0].set_xlabel('Epoch'); ax[0].grid(True)
ax[1].plot(history['val_iou'],   'g'); ax[1].set_title('Val mIoU');  ax[1].set_xlabel('Epoch'); ax[1].grid(True)
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'training_curves.png'), dpi=120)
plt.show()

In [ ]:
# ── Cell 13 : export ONNX ─────────────────────────────────────────────────────
import onnx

ckpt = torch.load(BEST_CKPT, map_location='cpu')
m    = lraspp_mobilenet_v3_large(num_classes=NUM_CLASSES)
m.load_state_dict(ckpt['model_state_dict'])
m.eval()

ONNX_OUT = MODEL_DIR / 'drivable_idd_lraspp_768x432.onnx'
torch.onnx.export(
    m, torch.zeros(1, 3, IMG_H, IMG_W), str(ONNX_OUT),
    input_names=['input'], output_names=['out'], opset_version=17,
    dynamic_axes={'input': {0: 'batch'}, 'out': {0: 'batch'}}
)
onnx.checker.check_model(str(ONNX_OUT))
print(f'✅ ONNX saved: {ONNX_OUT}')

In [ ]:
# ── Cell 14 : sanity check ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

m.eval()
sample_img  = np.array(Image.open(sorted(IMG_OUT.glob('*.jpg'))[0]).convert('RGB'))
sample_mask = np.array(Image.open(sorted(MSK_OUT.glob('*.png'))[0]))
inp = val_transform(image=sample_img)['image'].unsqueeze(0)
with torch.no_grad():
    pred = m(inp)['out'].argmax(1).squeeze().numpy()

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].imshow(sample_img);  ax[0].set_title('Input');        ax[0].axis('off')
ax[1].imshow(sample_mask, cmap='tab10', vmin=0, vmax=2); ax[1].set_title('Ground Truth'); ax[1].axis('off')
ax[2].imshow(pred,        cmap='tab10', vmin=0, vmax=2); ax[2].set_title('Prediction');   ax[2].axis('off')
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'sample_prediction.png'), dpi=120)
plt.show()

In [ ]:
# ── Cell 15 : list outputs ────────────────────────────────────────────────────
print('📁 Files to download (Output tab → right sidebar):')
for f in sorted(WORK_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(WORK_DIR)):<55} {f.stat().st_size/1e6:.1f} MB')
print('\n✅ Key outputs:')
print('   models/best.pth                        ← replace drivable_idd_full_best.pth locally')
print('   models/drivable_idd_lraspp_768x432.onnx ← replace ONNX locally')